In [2]:
import nibabel as nib
import numpy as np
from scipy.ndimage import center_of_mass

pet_path = "/home/administrator/Secretária/castor_analysis/CASTOR_FINAL_tests/CASTOR_CFCASTOR_CF_it3.nii"
seg_path = "/home/administrator/Champalimaud dados/Imagens/liver.nii"
out_path = "/home/administrator/Secretária/castor_analysis/CASTOR_FINAL_tests/CASTOR_CFCASTOR_CF_it3_aligned.nii"

# Load images
pet = nib.load(pet_path)
seg = nib.load(seg_path)

pet_data = pet.get_fdata()
seg_affine = seg.affine.copy()

# Flip PET along x axis (axis 0 because shape shown was (125,122,93))
pet_data_flipped = pet_data[::-1, :, :]

# Create new NIfTI with segmentation affine (so PET voxels map to segmentation world)
new_pet = nib.Nifti1Image(pet_data_flipped, seg_affine, header=pet.header)

# Update header sform/qform to match new affine (optional but recommended)
new_pet.set_sform(seg_affine)
new_pet.set_qform(seg_affine)

# Save
nib.save(new_pet, out_path)
print("Saved flipped & aligned PET to:", out_path)

# Quick verification (center-of-mass world coords for nonzero voxels)
def world_com_of_nonzero(img):
    data = img.get_fdata()
    nz = data != 0
    if not nz.any():
        return None
    com_vox = center_of_mass(nz.astype(np.uint8))
    com_vox = np.array(com_vox)
    # convert voxel index (i,j,k) to world using affine (append 1)
    world = img.affine.dot(np.append(com_vox, 1.0))[:3]
    return world

orig_pet_world = world_com_of_nonzero(pet)            # before flip, uses original affine
new_pet_world  = world_com_of_nonzero(new_pet)        # after flip+assign seg affine
seg_world      = world_com_of_nonzero(seg)            # segmentation COM

print("Segmentation COM world coords:", seg_world)
print("Original PET COM world coords (before):", orig_pet_world)
print("New PET COM world coords (after flip+assign):", new_pet_world)

Saved flipped & aligned PET to: /home/administrator/Secretária/castor_analysis/CASTOR_FINAL_tests/CASTOR_CFCASTOR_CF_it3_aligned.nii
Segmentation COM world coords: [ 73.58689381  -5.06735644 -22.46107438]
Original PET COM world coords (before): [ 0.30068328 -0.10031731  0.03031397]
New PET COM world coords (after flip+assign): [ 45.66588333  -6.74532158 -38.49768591]
